# Right Agent Group -- full stack + REAL CALL test on Kaggle GPU

**Two ways to use this notebook -- pick based on what you need:**

### A) Interactive (for a LIVE, clickable session -- use this to actually test the dashboard or place a call)
Open this notebook on kaggle.com yourself, in your own browser: Settings (right panel) -> Accelerator = **GPU T4 x2**, Internet = **ON**, then click **Run All** from the Kaggle UI.
Watch the output live -- when cell 9 prints the URLs, the container is still running, so open them immediately. The session stays alive until you close it or Kaggle's time limit hits -- no keep-alive cell needed.

### B) API push (Metadata file not found: kernel-metadata.json) -- for automated verification only, NOT for a live session
Kaggle runs API-pushed notebooks in **batch/commit mode**: the container tears down the instant the last cell finishes, and the output API only returns data *after* that -- so any URL you read this way is already dead. Use this mode only to confirm the pipeline builds and runs correctly (which it does), not to get a clickable link.

⚠️ This notebook reads all secrets from **Kaggle Secrets** (Add-ons -> Secrets in the notebook editor) -- it never embeds real credentials in the file itself. Before running, add these secrets (skip any you don't use yet):
`GITHUB_TOKEN`, `PG_PASSWORD`, `GOOGLE_CLIENT_ID`, `GOOGLE_CLIENT_SECRET`, `AUTH_SECRET`, `EXOTEL_SID`, `EXOTEL_API_KEY`, `EXOTEL_API_TOKEN`, `EXOTEL_CALLER_ID`, `EXOTEL_FLOW_APP_ID`, `WHATSAPP_SERVICE_KEY`, `SARVAM_API_KEY` (optional).

In [ ]:
%%bash
# ---- 1. System dependencies: Node 22, PostgreSQL, ffmpeg, cloudflared ----
set -e
curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
apt-get install -y -qq nodejs postgresql ffmpeg > /dev/null 2>&1
curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /usr/local/bin/cloudflared
echo "node: $(node -v) | psql: $(psql --version | awk '{print $3}') | ffmpeg + cloudflared OK"

In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess

# ---- 2. Clone the private repo + start PostgreSQL with matching password ----
secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
subprocess.run(
    ["git", "clone", "--quiet",
     f"https://{github_token}@github.com/arjungaming371-cmyk/right-agent-group.git",
     "/kaggle/working/app"],
    check=True,
)
print(subprocess.run(["git", "-C", "/kaggle/working/app", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
# ---- 3. Write .env for this session only (not stored in the repo) ----
# All real secrets come from Kaggle Secrets (Add-ons -> Secrets), never from this file.
import io
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def secret(name, default=""):
    try:
        return secrets.get_secret(name)
    except Exception:
        return default


PG_PASSWORD = secret("PG_PASSWORD")
GOOGLE_CLIENT_ID = secret("GOOGLE_CLIENT_ID")
GOOGLE_CLIENT_SECRET = secret("GOOGLE_CLIENT_SECRET")
AUTH_SECRET = secret("AUTH_SECRET")
EXOTEL_SID = secret("EXOTEL_SID")
EXOTEL_API_KEY = secret("EXOTEL_API_KEY")
EXOTEL_API_TOKEN = secret("EXOTEL_API_TOKEN")
EXOTEL_CALLER_ID = secret("EXOTEL_CALLER_ID")
EXOTEL_FLOW_APP_ID = secret("EXOTEL_FLOW_APP_ID")
WHATSAPP_SERVICE_KEY = secret("WHATSAPP_SERVICE_KEY")
SARVAM_API_KEY = secret("SARVAM_API_KEY")

ENV = f"""# ============================================================
# Right Agent Group FINAL_10 (Cloud API) — .env
# ❌ = must fill before starting   ⭕ = fill when going live
# ============================================================

# --- App URL (no trailing slash, must be https for OAuth + Meta webhook) ---
# LOCAL TESTING — switch to your https tunnel domain before going live
# (WhatsApp webhook + Exotel need https; Google login works on localhost)
NEXT_PUBLIC_APP_URL=http://localhost:3000

# --- PostgreSQL ---
PG_HOST=localhost
PG_PORT=5432
PG_DATABASE=right_agent_group
PG_USER=postgres
PG_PASSWORD={PG_PASSWORD}

# --- Ollama ---
OLLAMA_URL=http://localhost:11434
OLLAMA_MODEL=llama3.1:8b
OLLAMA_GPU=false                                     # IdeaPad Slim 3 = no NVIDIA GPU
OLLAMA_MAX_CONCURRENT=1                              # CPU-only: keep at 1

# --- Google Login (console.cloud.google.com/apis/credentials) ---
# Redirect URI must be exactly: https://your-domain.com/api/auth/google/callback
GOOGLE_CLIENT_ID={GOOGLE_CLIENT_ID}
GOOGLE_CLIENT_SECRET={GOOGLE_CLIENT_SECRET}
AUTH_SECRET={AUTH_SECRET}
ADMIN_EMAIL=arjun996625@gmail.com

# --- Exotel ---
EXOTEL_SID={EXOTEL_SID}
EXOTEL_API_KEY={EXOTEL_API_KEY}
EXOTEL_API_TOKEN={EXOTEL_API_TOKEN}
EXOTEL_SUBDOMAIN=api.exotel.com
EXOTEL_CALLER_ID={EXOTEL_CALLER_ID}                         # ExoPhone 095-138-86363
EXOTEL_FLOW_APP_ID={EXOTEL_FLOW_APP_ID}                           # aiagent23 Landing Flow (App Bazaar)

# --- WhatsApp Business Cloud API (SETUP-GUIDE-CLOUD-API.md steps 3,5,6) ---
WHATSAPP_TOKEN=                                      # ❌ STILL NEEDED — permanent System User token (EAA...)
WHATSAPP_PHONE_NUMBER_ID=                            # ❌ STILL NEEDED — Phone Number ID (not the phone number)
WHATSAPP_VERIFY_TOKEN=rag-verify-2026                # ❌ any string — must match Meta webhook setup
WHATSAPP_APP_SECRET=                                 # ❌ STILL NEEDED — REQUIRED for webhook signature verification
WHATSAPP_FORM_TEMPLATE=loan_application_form
# Auto-sent once per call (create + get these approved in WhatsApp Manager too):
WHATSAPP_CALL_FOLLOWUP_TEMPLATE=call_followup           # sent after any completed call with real conversation
WHATSAPP_MISSED_CALL_TEMPLATE=missed_call_followup      # sent when a call is missed/busy/no-answer

# --- Internal service auth (one shared secret for app ↔ voicebot ↔ STT) ---
WHATSAPP_SERVICE_KEY={WHATSAPP_SERVICE_KEY}
STT_API_KEY={WHATSAPP_SERVICE_KEY}

# --- STT (Whisper) ---
STT_SERVICE_URL=http://127.0.0.1:3003
STT_FORCE_DEVICE=cpu                                 # explicit: no CUDA on this laptop
# STT_MODEL=small                                    # default on CPU; uncomment to override

# --- TTS (Priya's voice) ---
# edge   = FREE Microsoft neural voices (default — zero cost per call)
# sarvam = paid, more natural Indic voices (needs SARVAM_API_KEY below)
# Coming: self-hosted MahaTTS on the client's GPU server (best of both)
TTS_PROVIDER=edge
# SARVAM_API_KEY={SARVAM_API_KEY}   # uncomment + set TTS_PROVIDER=sarvam to re-enable
# SARVAM_SPEAKER=priya                                # optional override; default: priya

# --- Voicebot ---
VOICEBOT_PORT=3002
APP_INTERNAL_URL=http://127.0.0.1:3000

# --- Cloudflare NAMED tunnel (stable URL — required for OAuth/Meta/Exotel) ---
CF_TUNNEL_NAME=rag                                   # ❌ create: cloudflared tunnel create rag

# --- Email confirmations (optional — leave blank to skip) ---
SMTP_HOST=
SMTP_PORT=465
SMTP_USER=
SMTP_PASS=
SMTP_FROM="Right Agent Group <yourbusiness@gmail.com>"

# --- Optional overrides ---
# APPLICATION_FORM_URL=                              # only if the form lives elsewhere
"""
io.open("/kaggle/working/app/.env", "w", encoding="utf-8").write(ENV)
print("wrote .env,", len(ENV), "chars — secrets loaded from Kaggle Secrets")

In [ ]:
%%bash
# ---- 4. Ollama on GPU + pull the model (~5 GB, the slow step) ----
# NOTE: the official curl|sh installer assumes systemd, which Kaggle
# containers do not have, so it fails silently. Installing the binary
# directly and running "ollama serve" ourselves avoids that entirely.
# Ollama ships releases as .tar.zst (zstd), not .tgz, as of v0.31.x —
# fetched from the GitHub release directly since ollama.com/download
# 404s for this asset name.
set -e
apt-get install -y -qq zstd > /dev/null 2>&1
curl -L -o /tmp/ollama.tar.zst https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst
tar --zstd -C /usr -xf /tmp/ollama.tar.zst
nohup ollama serve > /kaggle/working/ollama.log 2>&1 &
sleep 8
ollama pull llama3.1:8b
echo "--- GPU check ---"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%bash
# ---- 5. Install app dependencies (root + voicebot + STT) ----
set -e
cd /kaggle/working/app
npm install --silent 2>&1 | tail -1
cd server && npm install --silent 2>&1 | tail -1 && cd ..
pip install -q -r server/stt-service/requirements.txt
echo "All dependencies installed"

In [ ]:
# ---- 6. TWO public tunnels: website (3000) + voicebot WebSocket (3002) ----
import subprocess, re

def quick_tunnel(port):
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(90):
        line = p.stdout.readline()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m: return m.group(0)
    raise RuntimeError(f"tunnel for port {port} failed — re-run this cell")

web_url = quick_tunnel(3000)
vb_url  = quick_tunnel(3002)
wss_url = vb_url.replace("https://", "wss://") + "/voicebot"

open("/kaggle/working/tunnel_url.txt", "w").write(web_url)
open("/kaggle/working/wss_url.txt", "w").write(wss_url)
print("WEBSITE URL :", web_url)
print("VOICEBOT WSS:", wss_url)

In [ ]:
%%bash
# ---- 7. Point .env at the tunnel + GPU settings, then create the database ----
set -e
cd /kaggle/working/app
URL=$(cat /kaggle/working/tunnel_url.txt)
sed -i "s|^NEXT_PUBLIC_APP_URL=.*|NEXT_PUBLIC_APP_URL=${URL}|" .env
sed -i "s|^OLLAMA_GPU=.*|OLLAMA_GPU=true|" .env
sed -i "s|^STT_FORCE_DEVICE=.*|STT_FORCE_DEVICE=cuda|" .env
grep -q '^APP_INTERNAL_URL=' .env || echo 'APP_INTERNAL_URL=http://127.0.0.1:3000' >> .env
service postgresql start > /dev/null
PGPASS=$(grep -E '^PG_PASSWORD=' .env | cut -d= -f2 | awk '{print $1}')
sudo -u postgres psql -c "ALTER USER postgres PASSWORD '${PGPASS}';" > /dev/null
npm run db:setup && npm run db:check | tail -3

In [ ]:
%%bash
# ---- 8. Build + start everything (website, STT on GPU, voicebot) ----
cd /kaggle/working/app
npm run build 2>&1 | tail -3
nohup npm run start   > /kaggle/working/web.log      2>&1 &
cd server/stt-service
nohup python -m uvicorn app:app --host 127.0.0.1 --port 3003 > /kaggle/working/stt.log 2>&1 &
cd ..
nohup node voicebot-server.js > /kaggle/working/voicebot.log 2>&1 &
cd ..
# large-v3 on GPU needs real time to download + load — poll instead of a fixed sleep
for i in $(seq 1 24); do
  sleep 5
  READY=$(ss -tlnp | grep -cE ":(3000|3002|3003|11434) ")
  echo "[${i}0s] services up: $READY/4"
  if [ "$READY" = "4" ]; then break; fi
done
echo "--- listening ports (want 3000, 3002, 3003, 11434) ---"
ss -tlnp | grep -E ":(3000|3002|3003|11434)" | awk "{print \$4}"
echo "--- stt.log tail (if 3003 missing, this is why) ---"
tail -30 /kaggle/working/stt.log 2>&1

In [ ]:
# ---- 9. Login cookie + ALL your URLs and next steps ----
import base64, hashlib, hmac, json, time, re

env = open("/kaggle/working/app/.env", encoding="utf-8").read()
secret = re.search(r"^AUTH_SECRET=(\S+)", env, re.M).group(1)
b64 = lambda b: base64.urlsafe_b64encode(b).rstrip(b"=").decode()
payload = b64(json.dumps({"email": "kaggle-test@rightagentgroup.local", "exp": int(time.time()) + 86400}, separators=(",", ":")).encode())
sig = b64(hmac.new(secret.encode(), payload.encode(), hashlib.sha256).digest())

web = open("/kaggle/working/tunnel_url.txt").read().strip()
wss = open("/kaggle/working/wss_url.txt").read().strip()
print("=" * 60)
print("1. OPEN THE WEBSITE :", web)
print("   Press F12 -> Console -> paste this line -> Enter:")
print(f'   document.cookie="rag_session={payload}.{sig}; path=/"')
print("   Then go to:", web + "/dashboard")
print("=" * 60)
print("2. EXOTEL SETUP (once per Kaggle session):")
print("   my.exotel.com -> App Bazaar -> your flow (1288523)")
print("   -> Voicebot applet -> set URL to:")
print("  ", wss)
print("   -> Save the flow.")
print("=" * 60)
print("3. Then run the last cell to place the real test call.")

In [ ]:
%%bash
# ---- 10. Health + GPU speed test (no phone call yet) ----
cd /kaggle/working/app
KEY=$(grep -E '^WHATSAPP_SERVICE_KEY=' .env | cut -d= -f2 | awk '{print $1}')
echo '--- STT (expect model large-v3, device cuda) ---'
curl -s -H "x-api-key: $KEY" http://127.0.0.1:3003/health
echo ''
echo '--- Priya brain speed on GPU (was 10-25s on the laptop) ---'
time curl -s -X POST http://127.0.0.1:3000/api/calls/turn -H 'Content-Type: application/json' \
  -H "x-api-key: $KEY" -d '{"event":"turn","callSid":"KAGGLE-TEST-1","speech":"hello, I want a home loan","language":"english"}'

In [ ]:
# ---- 11. REAL PHONE CALL — read before running! ----
# Priya will actually dial the number below. Requirements:
#   - Exotel account KYC/trial-verified for this number
#   - Cell 9 step 2 done (wss URL saved in the Exotel flow THIS session)
# Uses trial credits. Change CONFIRM to True, then run.

CONFIRM = False
PHONE   = "+919908838090"   # your verified test number

import re, urllib.request, json as j
if not CONFIRM:
    print("Not calling. Set CONFIRM = True (after doing cell 9 step 2) and run again.")
else:
    env = open("/kaggle/working/app/.env", encoding="utf-8").read()
    import base64, hashlib, hmac, time
    secret = re.search(r"^AUTH_SECRET=(\S+)", env, re.M).group(1)
    b64 = lambda b: base64.urlsafe_b64encode(b).rstrip(b"=").decode()
    p = b64(j.dumps({"email": "kaggle-test@rightagentgroup.local", "exp": int(time.time()) + 3600}, separators=(",", ":")).encode())
    s = b64(hmac.new(secret.encode(), p.encode(), hashlib.sha256).digest())
    req = urllib.request.Request(
        "http://127.0.0.1:3000/api/calls",
        data=j.dumps({"phone": PHONE, "language": "telugu"}).encode(),
        headers={"Content-Type": "application/json", "Cookie": f"rag_session={p}.{s}"},
        method="POST",
    )
    try:
        print(urllib.request.urlopen(req, timeout=60).read().decode())
        print("\nPHONE SHOULD RING NOW. Answer it and talk to Priya!")
        print("Afterwards: check Voice Logs in the dashboard for the recording + transcript.")
    except urllib.error.HTTPError as e:
        print("Call failed:", e.read().decode())

## Optional: Svara-TTS (self-hosted, one voice for Hindi/Telugu/Indian English)

Run this ONLY on an already-running session (after the cells above have finished) -- it deploys
a second GPU service (Kenpath Svara-TTS via vLLM) alongside Ollama and Whisper.

WARNING - Experimental: this is a heavier, less-tested install than the rest of this notebook
(vLLM + flashinfer + torch, real CUDA-compatibility risk on Kaggle's specific image). Every step
below prints clear diagnostics so a failure is debuggable, not a silent dead end.

To actually use it for calls afterward: set TTS_PROVIDER=svara in .env, then re-run the
"restart the website" cell from earlier (git pull already has the code, just needs the env change).

In [ ]:
%%bash
# ---- Svara-TTS: check current GPU headroom before deploying a second service ----
nvidia-smi --query-gpu=index,memory.used,memory.total,memory.free --format=csv


In [ ]:
%%bash
# ---- Svara-TTS: clone + install (this is the slow, heavy step) ----
set -e
cd /kaggle/working
git clone --quiet https://github.com/Kenpath/svara-tts-inference.git
cd svara-tts-inference
echo "--- installing requirements (several minutes) ---"
pip install -q -r requirements.txt 2>&1 | tail -20
echo "--- done ---"


In [ ]:
%%bash
# ---- Svara-TTS: configure for a single T4 (fp8 quant, shorter context, conservative memory) ----
# Adjust CUDA_VISIBLE_DEVICES based on the nvidia-smi output above -- pick whichever GPU
# index has the most memory.free. Defaulting to GPU 1, assuming Ollama/Whisper favor GPU 0.
cd /kaggle/working/svara-tts-inference
cp .env.example .env 2>/dev/null || true
cat >> .env << 'ENVEOF'
VLLM_MODEL=kenpath/svara-tts-v1
VLLM_GPU_MEMORY_UTILIZATION=0.5
VLLM_MAX_MODEL_LEN=2048
VLLM_QUANTIZATION=fp8
VLLM_TENSOR_PARALLEL_SIZE=1
CUDA_VISIBLE_DEVICES=1
ENVEOF
echo "--- .env configured ---"
cat .env


In [ ]:
%%bash
# ---- Svara-TTS: start the server (poll instead of a fixed sleep -- model loading is slow) ----
cd /kaggle/working/svara-tts-inference
set -a; source .env; set +a
nohup python api/server.py > /kaggle/working/svara.log 2>&1 &
echo "started, pid $!"
for i in $(seq 1 24); do
  sleep 10
  if curl -s -o /dev/null --max-time 3 http://127.0.0.1:8080/v1/models 2>/dev/null; then
    echo "[${i}0s] server responding"
    break
  fi
  echo "[${i}0s] not up yet..."
done
echo "--- last 40 lines of svara.log ---"
tail -40 /kaggle/working/svara.log


In [ ]:
%%bash
# ---- Svara-TTS: real synthesis test (Telugu) -- proves it actually generates audio, not just "up" ----
cat > /tmp/svara_test_body.json << 'JSONEOF'
{"model":"svara-tts-v1","voice":"te_female","input":"నమస్కారం! నేను ప్రియ.","response_format":"wav"}
JSONEOF
curl -s -X POST http://127.0.0.1:8080/v1/audio/speech -H "Content-Type: application/json" -d @/tmp/svara_test_body.json -o /kaggle/working/svara-test.wav
echo "--- result ---"
ls -la /kaggle/working/svara-test.wav


In [ ]:
%%bash
# ---- Svara-TTS: switch the LIVE website over to it + restart ----
# Only run this AFTER cell 17 confirms svara-test.wav is a real, non-trivial
# audio file -- switching over to a TTS engine that isn't actually working
# would break every call, not just Svara ones.
set -e
cd /kaggle/working/app

SIZE=$(stat -c%s /kaggle/working/svara-test.wav 2>/dev/null || echo 0)
if [ "$SIZE" -lt 1000 ]; then
  echo "svara-test.wav is only ${SIZE} bytes -- looks like an error, not real audio."
  echo "Do NOT switch over. Check svara.log (cell 16 output) and re-run cell 17."
  exit 1
fi
echo "svara-test.wav looks real (${SIZE} bytes) -- switching over."

sed -i "s|^TTS_PROVIDER=.*|TTS_PROVIDER=svara|" .env
grep -q '^SVARA_URL=' .env || echo 'SVARA_URL=http://127.0.0.1:8080' >> .env

# Restart just the website process (voicebot-server.js picks up .env fresh on
# its next TTS call, no restart needed there -- only the Next.js process caches
# env at boot).
pkill -f "next start" 2>/dev/null || true
sleep 2
nohup npm run start > /kaggle/working/web.log 2>&1 &

for i in $(seq 1 12); do
  sleep 5
  if ss -tlnp | grep -q ':3000 '; then echo "[${i}0s] website back up on :3000"; break; fi
  echo "[${i}0s] waiting for website..."
done
echo "--- TTS_PROVIDER is now: ---"
grep '^TTS_PROVIDER=' .env